## 1. 定义 Pydantic Model 描述参数
如果函数的参数比较多而且比较复杂, 建议通过 pydantic model 来描述参数列表

In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
	location: str = Field(description="city name or coordinates")
	units: Literal["celsius","fahrenheit"] = Field(
		default="celsius",
		description="Temperature unit preference"
	)
	include_forecast: bool = Field(
		default=False,
		description="Include 5-day forecast"
	)

In [8]:
from langchain.tools import tool

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str="celsius", include_forecast: bool=False) -> str:
	"""Get current weather and optional forecast."""
	temp = 22 if units == "celsius" else 72
	result = f"current weather in {location}: {temp} degrees {units[0].upper()}"
	if include_forecast:
		result += "\nNext 5 days: Sunny"
	return result

In [9]:
get_weather.invoke({"location": "New York", "units": "fahrenheit", "include_forecast": True})

'current weather in New York: 72 degrees F\nNext 5 days: Sunny'

### 测试

In [10]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
	model="gpt-4o-mini",
	tools=[get_weather]
)

for token, metadata in agent.stream(
	{"messages": [HumanMessage(content="What is the weather in New York? Please provide the temperature in Fahrenheit and include the 5-day forecast.")]},
	stream_mode="messages"
):
	print(token.content, end="", flush=True)

current weather in New York: 72 degrees F
Next 5 days: SunnyThe current weather in New York is 72°F. 

Here's the 5-day forecast:
- **Day 1:** Sunny
- **Day 2:** Sunny
- **Day 3:** Sunny
- **Day 4:** Sunny
- **Day 5:** Sunny

It looks like you'll have a beautiful sunny week ahead!

## 2. 预定义工具 Tavily

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(
  max_results=3,
  topic="general"
)

### 方法一

In [ ]:
tavily.invoke("What is the capital of the Moon?")

### 方法二: Agent

In [ ]:
agent = create_agent(
	model="gpt-4o-mini",
	tools=[tavily],
	system_prompt="You are a helpful assistant that can answer questions using tools."
)

In [ ]:
response = agent.invoke(
	{"messages": [HumanMessage(content="What is the capital of the Moon?")]}
)

for message in response["messages"]:
	message.pretty_print()

### 方法三: 优化
先使用官方提供的做初始化, 再自己封装为 tool

In [ ]:
@tool
def web_search(query: str):
	"""Search the web for information"""
	return tavily.invoke(query)

# 定义结构化输出实体
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
	title: str = Field(description="The title of the web page cited in the answer")
	url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo(BaseModel):
	answer: str = Field(description="The final answer for user")
	reference: list[Reference] = Field(description="The web pages cited in the answer")

In [ ]:
agent = create_agent(
	model="gpt-4o-mini",
	tools=[web_search],
	system_prompt="You are a helpful assistant that can answer questions using tools.",
	response_format=AnswerInfo
)

response = agent.invoke(
	{"messages": [HumanMessage(content="What is the capital of the Moon?")]}
)

print(response["structured_response"])